##### SparkSession 

In [157]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("PropTech360-Test")
    .master("local[*]")
    .getOrCreate()
)

print(spark.version)

spark.stop()

4.2.0


In [158]:
# Environment check: record the versions in your README.
import sys
import importlib.metadata as metadata
print('Python:', sys.version.split()[0])
for package in ['pandas', 'numpy', 'pyspark', 'pyarrow', 'sqlalchemy', 'psycopg', 'boto3']:
    try:
        print(f'{package}: {metadata.version(package)}')
    except metadata.PackageNotFoundError:
        print(f'{package}: NOT INSTALLED (install before its assessed section)')

Python: 3.12.10
pandas: 3.0.6
numpy: 2.5.3
pyspark: 4.2.0
pyarrow: 25.0.1
sqlalchemy: 2.1.3
psycopg: 3.3.6
boto3: 1.43.108


In [159]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import pandas as pd

In [160]:
EXPECTED_COLUMNS = {
    "properties.csv": {
        "property_id",
        "property_name",
        "city"
    },

    "units.csv": {
        "unit_id",
        "property_id",
        "floor_area_sqm",
        "active"
    },

    "leases.csv": {
        "lease_id",
        "unit_id",
        "start_date",
        "end_date",
        "monthly_rent_usd"
    },

    "work_orders.csv": {
        "work_order_id",
        "unit_id",
        "opened_date",
        "closed_date",
        "priority",
        "response_hours",
        "cost_usd",
        "status"
    },

    "meter_readings.csv": {
        "reading_id",
        "unit_id",
        "reading_month",
        "energy_kwh"
    }
}

#### Verify the actual column names because your file may use name rather than proprty_name

In [161]:
def ingest_source(path, expected_columns):
    if not path.exists():
        raise FileNotFoundError(...)

    df = pd.read_csv(path)

    if df.empty:
        raise ValueError(...)

    missing_columns = expected_columns - set(df.columns)

    if missing_columns:
        raise ValueError(...)

    # hash file
    # create timestamp
    # return dataframe + manifest information

In [162]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import pandas as pd


def ingest_source(path: Path, expected_columns: set):
    """
    Validate a CSV source file, load it with stable string dtypes,
    and return both the DataFrame and ingestion metadata.
    """

    # 1. Validate that the file exists
    if not path.exists():
        raise FileNotFoundError(
            f"Source file does not exist: {path}"
        )

    # 2. Validate that the path is actually a file
    if not path.is_file():
        raise ValueError(
            f"Expected a file but received: {path}"
        )

    # 3. Validate that the file is not empty
    if path.stat().st_size == 0:
        raise ValueError(
            f"Source file is empty: {path}"
        )

    # 4. Load using stable dtypes
    # Keep source values as strings during ingestion.
    # Numeric/date conversion should happen later during validation/cleaning.
    try:
        df = pd.read_csv(
            path,
            dtype="string",
            keep_default_na=True
        )
    except Exception as exc:
        raise ValueError(
            f"Could not read CSV file {path.name}: {exc}"
        ) from exc

    # 5. Validate that records exist
    if df.empty:
        raise ValueError(
            f"Source file contains no data rows: {path}"
        )

    # 6. Validate required columns
    actual_columns = set(df.columns)

    missing_columns = expected_columns - actual_columns

    if missing_columns:
        raise ValueError(
            f"{path.name} is missing required columns: "
            f"{sorted(missing_columns)}"
        )

    # 7. Compute SHA-256 hash of the original file
    sha256 = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(8192), b""):
            sha256.update(chunk)

    file_hash = sha256.hexdigest()

    # 8. Create UTC ingestion timestamp
    ingest_timestamp = datetime.now(timezone.utc).isoformat()

    # 9. Create manifest record
    manifest_entry = {
        "filename": path.name,
        "file_path": str(path),
        "row_count": len(df),
        "column_count": len(df.columns),
        "ingest_timestamp_utc": ingest_timestamp,
        "sha256": file_hash
    }

    return df, manifest_entry

In [163]:
datasets = {}
ingestion_manifest = []

In [164]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import pandas as pd


def ingest_source(path, expected_columns):
    """
    Load a CSV source file, validate required columns,
    and create ingestion metadata.
    """

    path = Path(path)

    # 1. Check file exists
    if not path.exists():
        raise FileNotFoundError(
            f"Source file does not exist: {path.resolve()}"
        )

    # 2. Read CSV
    df = pd.read_csv(path)

    # 3. Validate expected columns
    missing_columns = set(expected_columns) - set(df.columns)

    if missing_columns:
        raise ValueError(
            f"{path.name} is missing required columns: "
            f"{sorted(missing_columns)}"
        )

    # 4. Calculate SHA-256 hash
    sha256 = hashlib.sha256(path.read_bytes()).hexdigest()

    # 5. Create ingestion manifest
    manifest = {
        "file_name": path.name,
        "file_path": str(path.resolve()),
        "row_count": len(df),
        "column_count": len(df.columns),
        "sha256": sha256,
        "ingested_at_utc": datetime.now(timezone.utc).isoformat()
    }

    return df, manifest

#### Test one of the actual file

In [165]:
RAW_DIR = Path(r"..data\raw\properties.csv")

In [166]:
RAW_DIR = Path("..") / "data" / "raw"

In [167]:
df, manifest = ingest_source(
    RAW_DIR / "properties.csv",
    {"property_id", "property_name", "city"}
)

In [168]:
print(df.head())

  property_id        property_name           city
0        P001  MetroSpace Tower 01          Lagos
1        P002  MetroSpace Tower 02          Abuja
2        P003  MetroSpace Tower 03         Ibadan
3        P004  MetroSpace Tower 04  Port Harcourt
4        P005  MetroSpace Tower 05          Lagos


In [169]:
print(manifest)

{'file_name': 'properties.csv', 'file_path': 'C:\\Documents\\Desktop\\PropTech360_Capstone_data_pipeline\\data\\raw\\properties.csv', 'row_count': 12, 'column_count': 3, 'sha256': 'c9865bcf2b142026c5dd05547f9c14d3e11f5a27a3e22bca52f5efe0c795ef19', 'ingested_at_utc': '2026-10-05T19:42:49.953277+00:00'}


In [170]:
from pathlib import Path

for file in Path(".").rglob("properties.csv"):
    print(file.resolve())

In [171]:
RAW_DIR = Path("data/raw")

In [172]:
{
    "filename": "properties.csv",
    "file_path": "data/raw/properties.csv",
    "row_count": 12,
    "column_count": 3,
    "ingest_timestamp_utc": "2026-10-03T...",
    "sha256": "..."
}

{'filename': 'properties.csv',
 'file_path': 'data/raw/properties.csv',
 'row_count': 12,
 'column_count': 3,
 'ingest_timestamp_utc': '2026-10-03T...',
 'sha256': '...'}

In [173]:
from pathlib import Path

RAW_DIR = Path("..") / "data" / "raw"

In [174]:
datasets = {}
ingestion_manifest = []

for filename, expected_columns in EXPECTED_COLUMNS.items():

    file_path = RAW_DIR / filename

    df, manifest_entry = ingest_source(
        file_path,
        expected_columns
    )

    datasets[filename] = df
    ingestion_manifest.append(manifest_entry)

In [175]:
print("Current working directory:")
print(Path.cwd())

print("\nRaw directory:")
print(RAW_DIR.resolve())

print("\nProperties path:")
print((RAW_DIR / "properties.csv").resolve())

print("\nDoes it exist?")
print((RAW_DIR / "properties.csv").exists())

Current working directory:
c:\Documents\Desktop\PropTech360_Capstone_data_pipeline\notebook

Raw directory:
C:\Documents\Desktop\PropTech360_Capstone_data_pipeline\data\raw

Properties path:
C:\Documents\Desktop\PropTech360_Capstone_data_pipeline\data\raw\properties.csv

Does it exist?
True


In [176]:
for filename, df in datasets.items():
    print(filename, df.shape)

properties.csv (12, 3)
units.csv (241, 4)
leases.csv (198, 5)
work_orders.csv (1501, 8)
meter_readings.csv (1440, 4)


In [177]:
import pandas as pd

test_df = pd.read_csv(RAW_DIR / "properties.csv")
print(test_df.columns.tolist())

['property_id', 'property_name', 'city']


In [178]:
ingestion_manifest

[{'file_name': 'properties.csv',
  'file_path': 'C:\\Documents\\Desktop\\PropTech360_Capstone_data_pipeline\\data\\raw\\properties.csv',
  'row_count': 12,
  'column_count': 3,
  'sha256': 'c9865bcf2b142026c5dd05547f9c14d3e11f5a27a3e22bca52f5efe0c795ef19',
  'ingested_at_utc': '2026-10-05T19:42:50.098543+00:00'},
 {'file_name': 'units.csv',
  'file_path': 'C:\\Documents\\Desktop\\PropTech360_Capstone_data_pipeline\\data\\raw\\units.csv',
  'row_count': 241,
  'column_count': 4,
  'sha256': 'f9bcef3d08d4662c270bfff385299306995ed6ba3f0c54e6b3754274a53c2748',
  'ingested_at_utc': '2026-10-05T19:42:50.101544+00:00'},
 {'file_name': 'leases.csv',
  'file_path': 'C:\\Documents\\Desktop\\PropTech360_Capstone_data_pipeline\\data\\raw\\leases.csv',
  'row_count': 198,
  'column_count': 5,
  'sha256': 'b6a8e10a11dfbca4358d5a9d60ad4709bc6511417137aeb58e389b21e31ebe0a',
  'ingested_at_utc': '2026-10-05T19:42:50.104546+00:00'},
 {'file_name': 'work_orders.csv',
  'file_path': 'C:\\Documents\\Deskto

#### Save the Manifest

In [179]:
import json
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
CURATED_DIR = PROJECT_ROOT / "data" / "curated"

CURATED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

manifest_path = CURATED_DIR / "ingestion_manifest.json"

with manifest_path.open(
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        ingestion_manifest,
        file,
        indent=4,
        default=str
    )

print(f"Manifest saved to: {manifest_path.resolve()}")

Manifest saved to: C:\Documents\Desktop\PropTech360_Capstone_data_pipeline\data\curated\ingestion_manifest.json


In [180]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
CURATED_DIR = PROJECT_ROOT / "data" / "curated"

RAW_DIR.mkdir(parents=True, exist_ok=True)
CURATED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw directory:", RAW_DIR)
print("Curated directory:", CURATED_DIR)

Project root: c:\Documents\Desktop\PropTech360_Capstone_data_pipeline
Raw directory: c:\Documents\Desktop\PropTech360_Capstone_data_pipeline\data\raw
Curated directory: c:\Documents\Desktop\PropTech360_Capstone_data_pipeline\data\curated


In [181]:
datasets.keys()

dict_keys(['properties.csv', 'units.csv', 'leases.csv', 'work_orders.csv', 'meter_readings.csv'])

In [182]:
datasets = {
    "properties": datasets["properties.csv"],
    "units": datasets["units.csv"],
    "leases": datasets["leases.csv"],
    "work_orders": datasets["work_orders.csv"],
    "meter_readings": datasets["meter_readings.csv"]
}

In [183]:
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

In [184]:
for name, df in datasets.items():
    print(name, df.shape)

properties (12, 3)
units (241, 4)
leases (198, 5)
work_orders (1501, 8)
meter_readings (1440, 4)


#### Creating identifier

In [185]:
IDENTIFIERS = {
    "properties.csv": "property_id",
    "units.csv": "unit_id",
    "leases.csv": "lease_id",
    "work_orders.csv": "work_order_id",
    "meter_readings.csv": "meter_reading_id"
}

#### Profile_table function

In [186]:
import pandas as pd


def profile_table(df: pd.DataFrame, identifier: str) -> dict:
    """
    Generate basic quality statistics for a dataframe.

    Returns:
        - row count
        - column count
        - null values by column
        - duplicate identifier count
        - pandas data types
    """

    profile = {
        "row_count": len(df),

        "column_count": len(df.columns),

        "nulls_by_column": df.isna().sum().to_dict(),

        "duplicate_identifier_count": (
            df.duplicated(subset=[identifier]).sum()
            if identifier in df.columns
            else None
        ),

        "data_types": {
            column: str(dtype)
            for column, dtype in df.dtypes.items()
        }
    }

    return profile

#### Test profiling function

In [187]:
properties_profile = profile_table(
    datasets["properties"],
    "property_id"
)

In [188]:
properties_profile

{'row_count': 12,
 'column_count': 3,
 'nulls_by_column': {'property_id': 0, 'property_name': 0, 'city': 0},
 'duplicate_identifier_count': np.int64(0),
 'data_types': {'property_id': 'str', 'property_name': 'str', 'city': 'str'}}

#### Profile all the five datasets

In [189]:
identifier = "properties_id"

properties_profile = profile_table(
    datasets["properties"],
    identifier
)

In [190]:
identifier = "leases_id"

leases_profile = profile_table(
    datasets["leases"],
    identifier
)

In [191]:
identifier = "unit_id"

units_profile = profile_table(
    datasets["units"],
    identifier
)

In [192]:
identifier = "meter_reading_id"

meter_readings_profile = profile_table(
    datasets["meter_readings"],
    identifier
)

In [193]:
identifier = "work_order_id"

work_orders_profile = profile_table(
    datasets["work_orders"],
    identifier
)

In [194]:
for filename, profile in quality_profiles.items():

    print("=" * 60)
    print("Leases")

    print("Rows:", profile["row_count"])
    print("Duplicate IDs:",
          profile["duplicate_identifier_count"])

    print("Nulls:")
    print(profile["nulls_by_column"])

In [195]:
for filename, profile in quality_profiles.items():

    print("=" * 60)
    print("meter_readings")

    print("Rows:", profile["row_count"])
    print("Duplicate IDs:",
          profile["duplicate_identifier_count"])

    print("Nulls:")
    print(profile["nulls_by_column"])

In [196]:
for filename, profile in quality_profiles.items():

    print("=" * 60)
    print("properties")

    print("Rows:", profile["row_count"])
    print("Duplicate IDs:",
          profile["duplicate_identifier_count"])

    print("Nulls:")
    print(profile["nulls_by_column"])

In [197]:
for filename, profile in quality_profiles.items():

    print("=" * 60)
    print("units")

    print("Rows:", profile["row_count"])
    print("Duplicate IDs:",
          profile["duplicate_identifier_count"])

    print("Nulls:")
    print(profile["nulls_by_column"])

In [198]:
for filename, profile in quality_profiles.items():

    print("=" * 60)
    print("work_orders")

    print("Rows:", profile["row_count"])
    print("Duplicate IDs:",
          profile["duplicate_identifier_count"])

    print("Nulls:")
    print(profile["nulls_by_column"])

#### Helper for Quarantine reasons

In [199]:
def add_rejection_reason(
    df: pd.DataFrame,
    mask: pd.Series,
    reason: str
):
    """
    Add a rejection reason to rows matching mask.
    """

    df.loc[mask, "rejection_reason"] = (
        df.loc[mask, "rejection_reason"]
        .fillna("")
        .apply(
            lambda existing:
            f"{existing}; {reason}".strip("; ")
            if existing
            else reason
        )
    )

#### Validate and Quarantine

In [200]:
def validate_and_quarantine(
    datasets: dict,
    run_id: str
):

    clean_datasets = {}

    quarantine_datasets = {}

    quality_summary = {}

    return (
        clean_datasets,
        quarantine_datasets,
        quality_summary
    )

#### Validate properties data

In [201]:
properties = datasets["properties"].copy()

properties["rejection_reason"] = pd.NA

##### checking for duplicate (properties)

In [202]:
duplicate_property = properties.duplicated(
    subset=["property_id"],
    keep=False
)

add_rejection_reason(
    properties,
    duplicate_property,
    "duplicate_property_id"
)

#### checking for missing ID

In [203]:
missing_property_id = properties["property_id"].isna()

add_rejection_reason(
    properties,
    missing_property_id,
    "missing_property_id"
)

#### Validate units data

In [204]:
units = datasets["units"].copy()

units["rejection_reason"] = pd.NA

#### checking for duplicate (units)

In [205]:
duplicate_unit = units.duplicated(
    subset=["unit_id"],
    keep=False
)

add_rejection_reason(
    units,
    duplicate_unit,
    "duplicate_unit_id"
)

#### check unit (property foreign key)

In [206]:
valid_property_ids = set(
    properties.loc[
        properties["rejection_reason"].isna(),
        "property_id"
    ]
)

#### Invalid property references

In [207]:
unknown_property = (
    ~units["property_id"].isin(valid_property_ids)
)

add_rejection_reason(
    units,
    unknown_property,
    "unknown_property_id"
)

#### validate numeric floor area

In [208]:
units["floor_area_sqm"] = pd.to_numeric(
    units["floor_area_sqm"],
    errors="coerce"
)

In [209]:
invalid_floor_area = (
    units["floor_area_sqm"].isna()
    |
    (units["floor_area_sqm"] <= 0)
)

add_rejection_reason(
    units,
    invalid_floor_area,
    "invalid_floor_area"
)

#### Validate leases

In [210]:
leases = datasets["leases"].copy()

leases["rejection_reason"] = pd.NA

#### convert to dates & rent

In [211]:
leases["start_date"] = pd.to_datetime(
    leases["start_date"],
    errors="coerce"
)

leases["end_date"] = pd.to_datetime(
    leases["end_date"],
    errors="coerce"
)

In [212]:
leases["monthly_rent_usd"] = pd.to_numeric(
    leases["monthly_rent_usd"],
    errors="coerce"
)

#### Detect bad lease dates

In [213]:
bad_start_date = leases["start_date"].isna()

add_rejection_reason(
    leases,
    bad_start_date,
    "invalid_start_date"
)

#### For end dates, you might allow missing end dates for active leases. But if an end date exists and it is before the start

In [214]:
bad_date_order = (
    leases["end_date"].notna()
    &
    leases["start_date"].notna()
    &
    (leases["end_date"] < leases["start_date"])
)

add_rejection_reason(
    leases,
    bad_date_order,
    "end_date_before_start_date"
)

#### Unit foreign key validation (Lease)

In [215]:
valid_unit_ids = set(
    units.loc[
        units["rejection_reason"].isna(),
        "unit_id"
    ]
)

In [216]:
unknown_lease_unit = (
    ~leases["unit_id"].isin(valid_unit_ids)
)

add_rejection_reason(
    leases,
    unknown_lease_unit,
    "unknown_unit_id"
)

#### Validate rent

In [217]:
invalid_rent = (
    leases["monthly_rent_usd"].isna()
    |
    (leases["monthly_rent_usd"] < 0)
)

add_rejection_reason(
    leases,
    invalid_rent,
    "invalid_monthly_rent"
)

#### check duplicate lease IDs

In [218]:
duplicate_lease = leases.duplicated(
    subset=["lease_id"],
    keep=False
)

add_rejection_reason(
    leases,
    duplicate_lease,
    "duplicate_lease_id"
)

#### Validate work orders

In [219]:
work_orders = datasets["work_orders"].copy()

work_orders["rejection_reason"] = pd.NA

In [220]:
duplicate_work_order = work_orders.duplicated(
    subset=["work_order_id"],
    keep=False
)

add_rejection_reason(
    work_orders,
    duplicate_work_order,
    "duplicate_work_order_id"
)

#### Validate work-order unit

In [221]:
unknown_work_order_unit = (
    ~work_orders["unit_id"].isin(valid_unit_ids)
)

add_rejection_reason(
    work_orders,
    unknown_work_order_unit,
    "unknown_unit_id"
)

##### validate response hours

In [225]:
work_orders["response_hours"] = pd.to_numeric(
    work_orders["response_hours"],
    errors="coerce"
)

In [226]:
missing_response = (
    work_orders["response_hours"].isna()
)

add_rejection_reason(
    work_orders,
    missing_response,
    "missing_response_hours"
)

#### checking for negative hours

In [227]:
negative_response = (
    work_orders["response_hours"] < 0
)

add_rejection_reason(
    work_orders,
    negative_response,
    "negative_response_hours"
)

#### Validate work-order date

In [231]:
date_column = "created_date"

if date_column in work_orders.columns:
    work_orders[date_column] = pd.to_datetime(
        work_orders[date_column],
        errors="coerce"
    )
else:
    print(f"Column '{date_column}' not found.")
    print("Available columns:", work_orders.columns.tolist())

Column 'created_date' not found.
Available columns: ['work_order_id', 'unit_id', 'opened_date', 'closed_date', 'priority', 'response_hours', 'cost_usd', 'status', 'rejection_reason']


In [229]:
print(datasets["work_orders"].columns.tolist())

['work_order_id', 'unit_id', 'opened_date', 'closed_date', 'priority', 'response_hours', 'cost_usd', 'status']


#### Validate meter-readings

In [234]:
meter_readings = datasets[
    "meter_readings"
].copy()

meter_readings["rejection_reason"] = pd.NA

In [236]:
meter_readings["energy_kwh"] = pd.to_numeric(
    meter_readings["energy_kwh"],
    errors="coerce"
)

#### Reject invalid numeric values

In [237]:
invalid_kwh = meter_readings["energy_kwh"].isna()

add_rejection_reason(
    meter_readings,
    invalid_kwh,
    "invalid_kwh"
)

#### Detecting negative energy_kwh

In [239]:
negative_kwh = meter_readings["energy_kwh"] < 0

add_rejection_reason(
    meter_readings,
    negative_kwh,
    "negative_kwh"
)


####  checking unit foreign key (metering_reading)

In [240]:
unknown_meter_unit = (
    ~meter_readings["unit_id"].isin(valid_unit_ids)
)

add_rejection_reason(
    meter_readings,
    unknown_meter_unit,
    "unknown_unit_id"
)

#### validate meter_month

In [242]:
print(datasets["meter_readings"].columns.tolist())

['reading_id', 'unit_id', 'reading_month', 'energy_kwh']


In [243]:
meter_readings["reading_month"] = pd.to_datetime(
    meter_readings["reading_month"],
    errors="coerce"
)

In [244]:
bad_meter_date = (
    meter_readings["reading_month"].isna()
)

add_rejection_reason(
    meter_readings,
    bad_meter_date,
    "invalid_reading_date"
)

##### cleaning vs quarantine records

In [245]:
def split_clean_and_quarantine(df):

    clean = df[
        df["rejection_reason"].isna()
    ].copy()

    quarantine = df[
        df["rejection_reason"].notna()
    ].copy()

    clean = clean.drop(
        columns=["rejection_reason"]
    )

    return clean, quarantine

#### Validate and quarantine () function

In [ ]:
import pandas as pd


def add_rejection_reason(
    df: pd.DataFrame,
    mask: pd.Series,
    reason: str
):

    current = df.loc[
        mask,
        "rejection_reason"
    ].fillna("")

    df.loc[
        mask,
        "rejection_reason"
    ] = current.apply(
        lambda x:
        f"{x}; {reason}".strip("; ")
        if x
        else reason
    )


def split_clean_and_quarantine(df):

    clean = df[
        df["rejection_reason"].isna()
    ].copy()

    quarantine = df[
        df["rejection_reason"].notna()
    ].copy()

    clean.drop(
        columns=["rejection_reason"],
        inplace=True
    )

    return clean, quarantine


def validate_and_quarantine(
    datasets: dict,
    run_id: str
):

    clean_datasets = {}
    quarantine_datasets = {}
    quality_summary = {}


    # ==================================
    # 1. PROPERTIES
    # ==================================

    properties = datasets[
        "properties.csv"
    ].copy()

    properties["rejection_reason"] = pd.NA


    duplicate_property = properties.duplicated(
        subset=["property_id"],
        keep=False
    )

    add_rejection_reason(
        properties,
        duplicate_property,
        "duplicate_property_id"
    )


    missing_property = (
        properties["property_id"].isna()
    )

    add_rejection_reason(
        properties,
        missing_property,
        "missing_property_id"
    )


    properties_clean, properties_bad = (
        split_clean_and_quarantine(properties)
    )


    valid_property_ids = set(
        properties_clean["property_id"]
    )


    # ==================================
    # 2. UNITS
    # =================================]

    units = datasets[
        "units.csv"
    ].copy()

    units["rejection_reason"] = pd.NA


    duplicate_unit = units.duplicated(
        subset=["unit_id"],
        keep=False
    )

    add_rejection_reason(
        units,
        duplicate_unit,
        "duplicate_unit_id"
    )


    unknown_property = (
        ~units["property_id"]
        .isin(valid_property_ids)
    )

    add_rejection_reason(
        units,
        unknown_property,
        "unknown_property_id"
    )


    units["floor_area_sqm"] = pd.to_numeric(
        units["floor_area_sqm"],
        errors="coerce"
    )


    invalid_floor_area = (
        units["floor_area_sqm"].isna()
        |
        (units["floor_area_sqm"] <= 0)
    )

    add_rejection_reason(
        units,
        invalid_floor_area,
        "invalid_floor_area"
    )


    units_clean, units_bad = (
        split_clean_and_quarantine(units)
    )


    valid_unit_ids = set(
        units_clean["unit_id"]
    )


    # ==================================
    # 3. LEASES
    # ==================================

    leases = datasets[
        "leases.csv"
    ].copy()

    leases["rejection_reason"] = pd.NA


    duplicate_lease = leases.duplicated(
        subset=["lease_id"],
        keep=False
    )

    add_rejection_reason(
        leases,
        duplicate_lease,
        "duplicate_lease_id"
    )


    leases["start_date"] = pd.to_datetime(
        leases["start_date"],
        errors="coerce"
    )


    leases["end_date"] = pd.to_datetime(
        leases["end_date"],
        errors="coerce"
    )


    leases["monthly_rent_usd"] = pd.to_numeric(
        leases["monthly_rent_usd"],
        errors="coerce"
    )


    invalid_start_date = (
        leases["start_date"].isna()
    )

    add_rejection_reason(
        leases,
        invalid_start_date,
        "invalid_start_date"
    )


    invalid_date_order = (
        leases["end_date"].notna()
        &
        leases["start_date"].notna()
        &
        (
            leases["end_date"]
            <
            leases["start_date"]
        )
    )

    add_rejection_reason(
        leases,
        invalid_date_order,
        "end_date_before_start_date"
    )


    unknown_unit = (
        ~leases["unit_id"]
        .isin(valid_unit_ids)
    )

    add_rejection_reason(
        leases,
        unknown_unit,
        "unknown_unit_id"
    )


    invalid_rent = (
        leases["monthly_rent_usd"].isna()
        |
        (
            leases["monthly_rent_usd"]
            < 0
        )
    )

    add_rejection_reason(
        leases,
        invalid_rent,
        "invalid_monthly_rent"
    )


    leases_clean, leases_bad = (
        split_clean_and_quarantine(leases)
    )


    # ==================================
    # 4. WORK ORDERS
    # ==================================

    work_orders = datasets[
        "work_orders.csv"
    ].copy()

    work_orders["rejection_reason"] = pd.NA


    duplicate_work_order = (
        work_orders.duplicated(
            subset=["work_order_id"],
            keep=False
        )
    )

    add_rejection_reason(
        work_orders,
        duplicate_work_order,
        "duplicate_work_order_id"
    )


    unknown_unit = (
        ~work_orders["unit_id"]
        .isin(valid_unit_ids)
    )

    add_rejection_reason(
        work_orders,
        unknown_unit,
        "unknown_unit_id"
    )


    # Adjust this name if required
    if "response_duration_hours" in work_orders.columns:

        work_orders[
            "response_duration_hours"
        ] = pd.to_numeric(
            work_orders[
                "response_duration_hours"
            ],
            errors="coerce"
        )


        missing_response = (
            work_orders[
                "response_duration_hours"
            ].isna()
        )

        add_rejection_reason(
            work_orders,
            missing_response,
            "missing_response_duration"
        )


        negative_response = (
            work_orders[
                "response_duration_hours"
            ] < 0
        )

        add_rejection_reason(
            work_orders,
            negative_response,
            "negative_response_duration"
        )


    # ==================================
    # 5. METER READINGS
    # ==================================

    meter_readings = datasets[
        "meter_readings.csv"
    ].copy()

    meter_readings[
        "rejection_reason"
    ] = pd.NA


    meter_readings["kwh"] = pd.to_numeric(
        meter_readings["kwh"],
        errors="coerce"
    )


    invalid_kwh = (
        meter_readings["kwh"].isna()
    )

    add_rejection_reason(
        meter_readings,
        invalid_kwh,
        "invalid_kwh"
    )


    negative_kwh = (
        meter_readings["kwh"] < 0
    )

    add_rejection_reason(
        meter_readings,
        negative_kwh,
        "negative_kwh"
    )


    unknown_unit = (
        ~meter_readings["unit_id"]
        .isin(valid_unit_ids)
    )

    add_rejection_reason(
        meter_readings,
        unknown_unit,
        "unknown_unit_id"
    )


    if "reading_date" in meter_readings.columns:

        meter_readings[
            "reading_date"
        ] = pd.to_datetime(
            meter_readings[
                "reading_date"
            ],
            errors="coerce"
        )


        bad_date = (
            meter_readings[
                "reading_date"
            ].isna()
        )

        add_rejection_reason(
            meter_readings,
            bad_date,
            "invalid_reading_date"
        )


    work_orders_clean, work_orders_bad = (
        split_clean_and_quarantine(
            work_orders
        )
    )

    meter_clean, meter_bad = (
        split_clean_and_quarantine(
            meter_readings
        )
    )


    # ==================================
    # STORE RESULTS
    # ==================================

    clean_datasets = {

        "properties.csv":
            properties_clean,

        "units.csv":
            units_clean,

        "leases.csv":
            leases_clean,

        "work_orders.csv":
            work_orders_clean,

        "meter_readings.csv":
            meter_clean
    }


    quarantine_datasets = {

        "properties.csv":
            properties_bad,

        "units.csv":
            units_bad,

        "leases.csv":
            leases_bad,

        "work_orders.csv":
            work_orders_bad,

        "meter_readings.csv":
            meter_bad
    }


    # ==================================
    # QUALITY SUMMARY
    # ==================================

    for filename in datasets.keys():

        source_rows = len(
            datasets[filename]
        )

        clean_rows = len(
            clean_datasets[filename]
        )

        rejected_rows = len(
            quarantine_datasets[filename]
        )

        quality_summary[filename] = {

            "run_id":
                run_id,

            "source_rows":
                source_rows,

            "clean_rows":
                clean_rows,

            "rejected_rows":
                rejected_rows,

            "reconciliation_check":
                source_rows
                ==
                clean_rows
                + rejected_rows
        }


    return (
        clean_datasets,
        quarantine_datasets,
        quality_summary
    )